In [7]:
import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
import os
from pathlib import Path
import matplotlib.pyplot as plt

save_path = "quarterly_PCE_PEQ_data"
data = {}
for file in os.listdir(save_path):
    if file.endswith(".pkl"):
        name = file[:-4]
        data[name] = pd.read_pickle(os.path.join(save_path,file))
        print(f"Loaded {name}")

bea_prices = data["bea_prices"]
bea_expenditures = data["bea_expenditures"]
delta_final = data["delta_final"]
products = data["products"]

Loaded delta_final
Loaded bea_prices
Loaded bea_expenditures
Loaded products


## CES additional data cleaning

Steps 1 - 3 and Basic pre-processing

In [8]:
"""Borrowing Tony's omega data, but change how 0s are cleaned and processed based on CES_Production_Take_Two by Nick"""
"""Is omega in step 2 mean the elements of A, is it related to omega in Tony's work? (Check with Nick)"""

delta_final_num = delta_final.apply(pd.to_numeric, errors="coerce")
delta_final_num.index   = delta_final_num.index.str.strip().str.lower()
delta_final_num.columns = delta_final_num.columns.str.strip().str.lower()
A_base = delta_final_num   # Ben's convention: columns are buyers, meaning they sum to gamma
gamma = delta_final_num.sum(axis=0) 
print("shape:", A_base.shape)

products = A_base.columns

A_base


shape: (233, 233)


,accessories and parts,agricultural machinery,air transportation,aircraft,alcohol in purchased meals,all other professional medical services,"amusement parks, campgrounds, and related recreational services","audio discs, tapes, vinyl, and permanent digital downloads",audio equipment,auto leasing,...,video and audio streaming and rental,"video discs, tapes, and permanent digital downloads",vocational rehabilitation services,watches,water supply and sewage maintenance,water transportation,window coverings,wine,women's and girls' clothing,workers' compensation
accessories and parts,0.116005,0.010955,0.000354,0.010554,0.001331,0.001512,0.003670,0.004066,0.007641,0.007630,...,0.004657,0.001385,0.011701,0.001242,0.002969,0.003259,0.003365,0.009876,0.001870,0.006625
agricultural machinery,0.000674,0.002214,0.000053,0.001033,0.000165,0.000107,0.000194,0.000244,0.000340,0.000150,...,0.000158,0.000064,0.000134,0.000106,0.001015,0.000128,0.000209,0.000627,0.000121,0.000025
air transportation,0.000389,0.000131,0.000067,0.000639,0.000190,0.000136,0.000280,0.000234,0.000302,0.000443,...,0.000488,0.000095,0.000185,0.000142,0.000139,0.000498,0.000138,0.000459,0.000131,0.000013
aircraft,0.000485,0.000127,0.000027,0.150861,0.000069,0.000089,0.000288,0.000162,0.000311,0.000113,...,0.000091,0.000244,0.000238,0.000058,0.000202,0.000064,0.000085,0.000342,0.000082,0.000020
alcohol in purchased meals,0.002395,0.000820,0.001053,0.001630,0.001646,0.002520,0.001490,0.001719,0.001857,0.001840,...,0.004365,0.001486,0.003894,0.000851,0.001167,0.001140,0.000739,0.002314,0.000622,0.000463
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
water transportation,0.000738,0.000183,0.003924,0.000507,0.000649,0.000150,0.000503,0.000404,0.000654,0.000260,...,0.000441,0.000195,0.000376,0.000135,0.000777,0.008463,0.000823,0.001020,0.000367,0.000061
window coverings,0.000414,0.000068,0.000020,0.000246,0.000110,0.000082,0.000147,0.000133,0.000224,0.000090,...,0.000097,0.000039,0.000542,0.000061,0.000158,0.000074,0.000470,0.000327,0.000076,0.000054
wine,0.000209,0.000071,0.000158,0.000161,0.001366,0.000112,0.000383,0.000105,0.000160,0.000095,...,0.000254,0.000057,0.000086,0.000084,0.000383,0.000092,0.000083,0.023310,0.000060,0.000010
women's and girls' clothing,0.001499,0.000167,0.000041,0.000410,0.000169,0.000173,0.000246,0.000281,0.000419,0.000126,...,0.000193,0.000093,0.000276,0.000101,0.000251,0.000140,0.000920,0.000712,0.000927,0.000199


In [9]:
price_panel = (bea_prices.pivot_table(index="date", columns="products", values="Prices", aggfunc="first").sort_index())
price_panel = price_panel.reindex(columns = products)
missing = set(products) - set(bea_prices["products"].unique())
print(len(missing), sorted(missing)[:10])

expend_panel = (bea_expenditures.pivot_table(index="date", columns="products", values="Expenditures", aggfunc="first"))
expend_panel = expend_panel.reindex(columns = products)           

# dates for these two datasets don't match, so we'll clean them up
has_p = price_panel.notna().any(axis=1)
has_e = expend_panel.fillna(0).sum(axis=1) > 0
sample_dates = price_panel.index[has_p & has_e.reindex(price_panel.index, fill_value=False)]

price_panel  = price_panel.loc[sample_dates].fillna(0)
expend_panel = expend_panel.reindex(sample_dates).fillna(0)

0 []


Check max col sum between 0 and 1

In [10]:
assert (A_base.index == A_base.columns).all()

col_sums = A_base.sum(axis=0)
row_sums = A_base.sum(axis=1)
print(col_sums.mean(), col_sums.max())
assert 0 < col_sums.max() < 1
assert np.allclose(col_sums.to_numpy(), gamma.reindex(products).to_numpy())

0.19248280978626212 0.7252477468940651


Replace non-existing prices

In [11]:
A_monthly = {}
for tm1, t in zip(price_panel.index[:-1], price_panel.index[1:]):
    exists_tm1 = price_panel.loc[tm1].gt(0).to_numpy()  
    existence_mask = np.outer(exists_tm1, exists_tm1) 
    A_monthly[t] = A_base.mul(existence_mask) # Starts from 1959-04-01, careful when referencing

## Production Function Derivation

4-5. Calculate Sales

In [12]:
# does BEA expenditure match PCE at all? - yes, it seems it has the same number of unique products/categories fortunately

sales_monthly = {}

for date in A_monthly.keys():
    pce_expenditure_t = (bea_expenditures.loc[bea_expenditures["date"] == date].set_index("products")["Expenditures"])
    A_t = A_monthly[date]
    I = np.eye(len(A_t))
    pce_expenditure_t = pce_expenditure_t.reindex(A_t.index, fill_value=0) 
    assert pce_expenditure_t.index.equals(A_t.index)

    sales_t = np.linalg.solve(I - A_t.to_numpy(), pce_expenditure_t.to_numpy()) # NS: removed .T
    # To get around Det = 0, we solve for "[I - A_t.T.to_numpy()] @ sales_t = PC_i,t" instead

    sales_monthly[date] = pd.Series(sales_t, index = A_t.index, name = date)

sales_monthly.keys()

dict_keys([Timestamp('1960-01-01 00:00:00'), Timestamp('1960-04-01 00:00:00'), Timestamp('1960-07-01 00:00:00'), Timestamp('1960-10-01 00:00:00'), Timestamp('1961-01-01 00:00:00'), Timestamp('1961-04-01 00:00:00'), Timestamp('1961-07-01 00:00:00'), Timestamp('1961-10-01 00:00:00'), Timestamp('1962-01-01 00:00:00'), Timestamp('1962-04-01 00:00:00'), Timestamp('1962-07-01 00:00:00'), Timestamp('1962-10-01 00:00:00'), Timestamp('1963-01-01 00:00:00'), Timestamp('1963-04-01 00:00:00'), Timestamp('1963-07-01 00:00:00'), Timestamp('1963-10-01 00:00:00'), Timestamp('1964-01-01 00:00:00'), Timestamp('1964-04-01 00:00:00'), Timestamp('1964-07-01 00:00:00'), Timestamp('1964-10-01 00:00:00'), Timestamp('1965-01-01 00:00:00'), Timestamp('1965-04-01 00:00:00'), Timestamp('1965-07-01 00:00:00'), Timestamp('1965-10-01 00:00:00'), Timestamp('1966-01-01 00:00:00'), Timestamp('1966-04-01 00:00:00'), Timestamp('1966-07-01 00:00:00'), Timestamp('1966-10-01 00:00:00'), Timestamp('1967-01-01 00:00:00'), Tim

In [13]:
broke = False
for date in sales_monthly:
    bad = sales_monthly[date] < 0
    if bad.any():
        print(date)
        print(sales_monthly[date][bad])
        broke = True
        break
# NS: Sorry, I like confirmations for and against :)
if broke == False:
    print('If you see this, sales are all positive')

If you see this, sales are all positive


6-8. Calculate B

In [14]:
epsilon = 0.3 # arbitrary number

pi_ela_monthly = {} 
pi_gross_monthly = {}

# Need to adjust price panel before calculating inflation, otherwise we run into divide by 0 NaN
# Again, Tony already adjusted data such that missing datapoints are set to 0
for tm1, t in zip(price_panel.index[:-1], price_panel.index[1:]):
    p_tm1 = price_panel.loc[tm1].copy()
    p_t = price_panel.loc[t].copy()

    exit = (p_tm1 != 0) & (p_t == 0)
    p_t.loc[exit] = p_tm1.loc[exit]

    enter = (p_tm1 == 0) & (p_t != 0)
    p_tm1.loc[enter] = p_t[enter]

    pi_t = (p_t / p_tm1) 

    # 0/0 = nan
    missing_both = pi_t.isna()
    pi_t[missing_both] = 1.0

    pi_ela_monthly[t] = pi_t ** (1 - epsilon)
    pi_gross_monthly[t] = pi_t

In [15]:
# double check that the 0/0 nan is caught correctly
# pi_check = {}
# for tm1, t in zip(price_panel.index[:-1], price_panel.index[1:]):
#     p_tm1 = price_panel.loc[tm1].copy()
#     p_t = price_panel.loc[t].copy()

#     exit = (p_tm1 != 0) & (p_t == 0)
#     p_t.loc[exit] = p_tm1.loc[exit]
    
#     enter = (p_tm1 == 0) & (p_t != 0)
#     p_tm1.loc[enter] = p_t[enter]

#     missing_both = (p_tm1 == 0) & (p_t == 0) 
#     p_tm1[missing_both],p_t[missing_both] = 1.0, 1.0

#     pi_t = (p_t / p_tm1) ** (1 - epsilon)
#     pi_check[t] = pi_t


# assert pi_check.keys() == pi_ela_monthly.keys()
# for date in pi_check:
#     assert pi_check[date].equals(pi_ela_monthly[date])

In [16]:
B_monthly = {}

for date in A_monthly.keys():
    A_t = A_monthly[date]
    I = np.eye(len(A_t))
 
    # separated so I don't get too dizzy (NS: sensible)
    Leontief_inv = I - A_t.T.to_numpy()     # NS: Added .T
    
    col_sums = A_t.to_numpy().sum(axis=0) 
    diag_matrix = np.diag(1 - col_sums)

    B_t = np.linalg.solve(Leontief_inv, diag_matrix)
    B_monthly[date] = B_t

B_monthly[next(iter(B_monthly))]




array([[5.29462862e-01, 9.44830228e-04, 6.95516921e-04, ...,
        2.27121298e-04, 1.84104778e-03, 1.24188035e-03],
       [6.30292089e-03, 8.94969299e-01, 1.96071427e-04, ...,
        6.35603578e-05, 2.23568477e-04, 3.31735539e-04],
       [5.11815767e-04, 7.21523326e-05, 9.38293256e-01, ...,
        1.11377162e-04, 6.47110423e-05, 1.51410808e-04],
       ...,
       [7.12942210e-03, 7.97794290e-04, 6.86814773e-04, ...,
        6.18376616e-01, 8.81380015e-04, 1.24029309e-03],
       [1.37161151e-03, 1.54926550e-04, 1.82177264e-04, ...,
        5.25326201e-05, 9.18631798e-01, 3.41428746e-04],
       [3.82284411e-03, 6.33895161e-05, 5.34464807e-05, ...,
        1.73232087e-05, 2.23993705e-04, 9.28613363e-01]], shape=(233, 233))

9. Check B is valid (row sums to 1)

In [17]:
broke = False
for date in B_monthly:
    row_sums = B_monthly[date] @ np.ones(len(B_monthly[date]))
    col_sums = B_monthly[date].sum(axis=0)
    
    bad = abs((row_sums) -1) > 0.00001 # all row sums positive so Abs is ok
    if bad.any():
        print("B_matrix stats for", date)
        print("Min row sum:", row_sums.min(), "\nMax row sum:", row_sums.max())
        print("Offending rows:")
        print(B_monthly[date][bad])
        broke = True
        break
if broke == False:
    print('If you see this, B rows sum to 1!')

If you see this, B rows sum to 1!


10 - 12. Calculate Value added inflation

In [18]:
pi_VA_monthly = {}

for t in B_monthly.keys():
    pi_ela_t = pi_ela_monthly[t]
    pi_VA_monthly[t] = np.linalg.solve(B_monthly[t], pi_ela_monthly[t])


In [19]:
# step 11 double checking
assert B_monthly.keys() == pi_VA_monthly.keys()
assert B_monthly.keys() == pi_ela_monthly.keys()
for date in B_monthly.keys():
    assert np.allclose(B_monthly[date] @ pi_VA_monthly[date], pi_ela_monthly[date])

Net Value Added Inflation

In [20]:
net_pi_VA_monthly_dict = {}

for time, value in pi_VA_monthly.items():
    net_pi_VA_monthly_dict[time] = pi_VA_monthly[time]**(1/(1-epsilon)) -1
# pi_VA_monthly no longer has negative values by using delta_final, so omega in CES_take three is different from Tony's?


# I totally forgot to use dataframes but maybe I just stick with the dictionaries at this point?
net_VA_monthly = pd.DataFrame(net_pi_VA_monthly_dict, index = products)
net_VA_monthly = net_VA_monthly.T # flip so that rows = dates, cols = products

net_VA_monthly

,accessories and parts,agricultural machinery,air transportation,aircraft,alcohol in purchased meals,all other professional medical services,"amusement parks, campgrounds, and related recreational services","audio discs, tapes, vinyl, and permanent digital downloads",audio equipment,auto leasing,...,video and audio streaming and rental,"video discs, tapes, and permanent digital downloads",vocational rehabilitation services,watches,water supply and sewage maintenance,water transportation,window coverings,wine,women's and girls' clothing,workers' compensation
1960-01-01,-0.005096,0.004105,0.016850,-0.001407,0.020543,0.000000,0.004425,-0.002081,-0.001668,0.000000,...,0.000000,0.000000,0.000000,-0.000029,0.008167,0.015376,0.023036,0.005047,0.000923,-0.021533
1960-04-01,-0.003590,0.011206,0.029026,-0.012097,-0.010774,0.000000,0.008629,0.000247,-0.005214,0.000000,...,0.000000,0.000000,0.000000,-0.001523,0.007043,0.028940,0.042452,-0.001673,0.004241,-0.005895
1960-07-01,0.004079,0.008064,0.011943,0.004138,0.005844,0.000000,0.006074,0.007210,-0.006640,0.000000,...,0.000000,0.000000,0.000000,0.001515,0.007670,-0.000119,0.031807,0.004188,0.000279,-0.007643
1960-10-01,-0.004141,0.007143,0.014474,-0.002858,-0.011552,0.000000,0.001737,-0.001929,-0.014222,0.000000,...,0.000000,0.000000,0.000000,-0.002416,0.009616,0.027289,0.011555,-0.002550,0.000413,0.002380
1961-01-01,-0.004410,0.001444,0.017137,0.009097,0.007955,0.000000,0.010272,0.000906,-0.001983,0.000000,...,0.000000,0.000000,0.000000,0.000937,0.007059,0.014042,0.008445,0.001590,-0.000438,0.004604
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-01-01,0.018193,0.002385,0.013483,0.007915,-0.002618,0.006738,0.013801,0.011233,0.005885,-0.022667,...,0.002848,0.054132,0.012068,0.035597,0.026384,-0.036156,0.007233,0.004857,-0.008915,-0.021457
2024-04-01,0.007632,0.001572,-0.020666,0.011001,0.004915,-0.004200,0.000905,0.010926,0.003776,0.024158,...,-0.016059,0.119486,0.011965,-0.003067,0.008547,0.010532,0.020346,-0.001388,0.021060,0.014686
2024-07-01,0.020152,-0.004682,-0.000861,0.009083,0.005026,0.009212,0.003736,0.017259,0.002173,-0.010377,...,0.060649,-0.014005,0.006331,-0.009916,0.013003,-0.016134,0.007895,-0.005587,0.006355,0.000842
2024-10-01,-0.007356,0.004235,0.042798,0.012163,0.015207,0.003629,0.015825,0.019064,-0.002206,0.003827,...,-0.016283,-0.027276,0.008266,0.047298,0.014280,0.008685,-0.010294,-0.009583,-0.015317,-0.006485


In [21]:
# shouldn't have NaNs but a quick check either way
n_nan_VA = net_VA_monthly.isna().sum().sum()
print(n_nan_VA)

0


In [22]:
# also a check here since we need to use sales_df
sales_df = pd.DataFrame(sales_monthly).T
sales_df = sales_df.reindex(columns=products)

print(sales_df.columns[:5])
print(sales_df.isna().all().sum())

Index(['accessories and parts', 'agricultural machinery', 'air transportation',
       'aircraft', 'alcohol in purchased meals'],
      dtype='object')
0


Clean up sales_df and net_VA_monthly before doing the VAR

In [23]:
sales_positive = sales_df.where(sales_df > 0)

log_sales = np.log(sales_df.where(sales_df > 0))
log_sales = log_sales.ffill()
sales_growth = log_sales.diff() #net_VA_monthly is inflation so it's already a growth rate

# should always pass, but it's here now
common_dates = net_VA_monthly.index.intersection(sales_growth.index)
net_VA_monthly = net_VA_monthly.loc[common_dates]
sales_growth = sales_growth.loc[common_dates]
print("missing dates:", len(net_VA_monthly.index) - len(common_dates))



missing dates: 0


12. VAR

In [24]:
lags = 4 # Tony did 4 quarters, I'll use 12 months for now? doublecheck with Nick on this ***
min_obs = 3 * lags

residual_frames = []
skipped = {}

for p in products:

    # all months for one product
    data_p = pd.DataFrame({"va_inflation": net_VA_monthly[p], "sales_growth": sales_growth[p]})
    data_p = data_p.replace([np.inf, -np.inf], np.nan)
    data_p = data_p.dropna()

    if len(data_p) < min_obs:
        skipped[p] = "not enough data points"

    if (data_p.std() == 0).any():
        skipped[p] = "constant series"

    model_p = VAR(data_p)
    result_p = model_p.fit(lags)

    resid_p = result_p.resid.copy()
    resid_p = resid_p.rename(columns={
        "va_inflation": "residual_va",
        "sales_growth": "residual_sales",
    })
    resid_p["product"] = p
    residual_frames.append(resid_p) 

CES_residuals = pd.concat(residual_frames)
CES_residuals = CES_residuals.rename_axis("date")
CES_residuals = CES_residuals.reset_index()
    

/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)
/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)
/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)
/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)
/home/leco/.local/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:480: ValueWarning: No frequency information was provided, so in

13. Classify demand and supply shocks

In [25]:
# same sign = demand shock, opposite = supply shock

residual_product = (CES_residuals["residual_va"] * CES_residuals["residual_sales"])

CES_residuals["D_CES"] = (residual_product >= 0).astype(int)

D_CES = CES_residuals.pivot(index = "date", columns = "product", values = "D_CES",)

In [26]:
print("total products:", CES_residuals["product"].nunique)
print("products skiped:", len(skipped))

total products: <bound method IndexOpsMixin.nunique of 0        accessories and parts
1        accessories and parts
2        accessories and parts
3        accessories and parts
4        accessories and parts
                 ...          
56437    workers' compensation
56438    workers' compensation
56439    workers' compensation
56440    workers' compensation
56441    workers' compensation
Name: product, Length: 56442, dtype: object>
products skiped: 0


In [27]:
share_demand = CES_residuals["D_CES"].mean()
print("share classified demand:", round(share_demand, 3))

print("D_CES shape:", D_CES.shape)
print("Net_VA_inflation shape:", net_VA_monthly.shape)

share classified demand: 0.547
D_CES shape: (256, 233)
Net_VA_inflation shape: (261, 233)


14. Calculate S_t

In [28]:
S_monthly = {} # need to be careful about orientation, currently rows = products, col = dates

for t in pi_ela_monthly.keys():
    S_t = (pi_gross_monthly[t] - 1)  / (pi_ela_monthly[t] - 1)
    
    # another divide by 0 case because we set pi_ela_monthly to 1 for so many data points D:
    # placeholder , code can handle some changes
    missing_value = S_t.isna()
    S_t[missing_value] = 1.0 

    S_monthly[t] = S_t
    
list(S_monthly.keys())[:5] #everything else is a dictionary anyways so I guess I stick with it

[Timestamp('1960-01-01 00:00:00'),
 Timestamp('1960-04-01 00:00:00'),
 Timestamp('1960-07-01 00:00:00'),
 Timestamp('1960-10-01 00:00:00'),
 Timestamp('1961-01-01 00:00:00')]

15. Calculate pi_i,t

In [29]:
# LHS = pi_i,t = P_i,t / P_i,t-1 = pi_gross_monthly 
# RHS = S_t * (pi_ela_monthly - 1)
for date in pi_gross_monthly.keys():
    assert np.allclose(pi_gross_monthly[date] - 1, S_monthly[date] * (pi_ela_monthly[date] - 1))
    # The equation does not hold with gross inflation, only net inflation with the extra -1 ***
    # If the next step works without problem I will continue with it then
    # clarify differene between pi in step 12 and 15, 16 ***


    # testing_difference = (pi_gross_monthly[date] - 1) - (S_monthly[date] * (pi_ela_monthly[date] - 1))
    # bad = abs(testing_difference) > 0.0001 
    # if bad.any():
    #     print(testing_difference[bad])
    #     break

16. pi_CPI,t

In [30]:
p_dates = pd.Index(price_panel.index)
e_dates = pd.Index(expend_panel.index)
print("price:", p_dates.min(), p_dates.max(), len(p_dates))
print("expend:", e_dates.min(), e_dates.max(), len(e_dates))
print("in prices, not expend:", p_dates.difference(e_dates)[:10])
print("in expend, not prices:", e_dates.difference(p_dates)[:10])
print(p_dates[:6].tolist(), e_dates[:6].tolist())   # check day-of-month conventions

price: 1959-10-01 00:00:00 2025-01-01 00:00:00 262
expend: 1959-10-01 00:00:00 2025-01-01 00:00:00 262
in prices, not expend: DatetimeIndex([], dtype='datetime64[ns]', name='date', freq=None)
in expend, not prices: DatetimeIndex([], dtype='datetime64[ns]', name='date', freq=None)
[Timestamp('1959-10-01 00:00:00'), Timestamp('1960-01-01 00:00:00'), Timestamp('1960-04-01 00:00:00'), Timestamp('1960-07-01 00:00:00'), Timestamp('1960-10-01 00:00:00'), Timestamp('1961-01-01 00:00:00')] [Timestamp('1959-10-01 00:00:00'), Timestamp('1960-01-01 00:00:00'), Timestamp('1960-04-01 00:00:00'), Timestamp('1960-07-01 00:00:00'), Timestamp('1960-10-01 00:00:00'), Timestamp('1961-01-01 00:00:00')]


In [31]:
monthly_totals = expend_panel.sum(axis=1)

omega_PCE_monthly = {}

for t in pi_gross_monthly.keys(): # the main operaiton for omega_PCE_monthly is to multiply with pi_gross_monthly, so they should match 
    omega_PCE_monthly[t] = expend_panel.loc[t] / expend_panel.loc[t].sum()

omega_PCE_monthly[next(iter(omega_PCE_monthly))]

accessories and parts          0.002903
agricultural machinery         0.000000
air transportation             0.002053
aircraft                       0.000000
alcohol in purchased meals     0.013773
                                 ...   
water transportation           0.000003
window coverings               0.002626
wine                           0.001751
women's and girls' clothing    0.040635
workers' compensation          0.001397
Name: 1960-01-01 00:00:00, Length: 233, dtype: float64

In [32]:
# caulate omega_CPI_monthly, same way as we did for bea prices

monthly_totals = expend_panel.sum(axis=1)

omega_PCE_monthly = {}

for t in pi_gross_monthly.keys(): # the main operaiton for omega_PCE_monthly is to multiply with pi_gross_monthly, so they should match 
    omega_PCE_monthly[t] = expend_panel.loc[t] / expend_panel.loc[t].sum()

omega_PCE_monthly[next(iter(omega_PCE_monthly))]

accessories and parts          0.002903
agricultural machinery         0.000000
air transportation             0.002053
aircraft                       0.000000
alcohol in purchased meals     0.013773
                                 ...   
water transportation           0.000003
window coverings               0.002626
wine                           0.001751
women's and girls' clothing    0.040635
workers' compensation          0.001397
Name: 1960-01-01 00:00:00, Length: 233, dtype: float64

17. Weights omega_VA,t

In [33]:
pi_CPI_monthly = {}

for t in pi_gross_monthly.keys():
    pi_CPI_monthly[t] = (omega_PCE_monthly[t] * (pi_gross_monthly[t] - 1)).sum()
    pi_CPI_check = omega_PCE_monthly[t].T @ (pi_gross_monthly[t] - 1)
    assert np.isclose(pi_CPI_monthly[t], pi_CPI_check)

In [34]:
omega_VA_monthly = {}

for t in pi_CPI_monthly.keys():
    A_t = A_monthly[t]
    I = np.eye(len(A_t))

    # not sure about how to do this one exactly ***
    # x' = w'PCE(I - A't)^-1 --> (I - A_t) x = w_PCE --> solve for x with linalg.solve
    # w_VA = xy, where y = diag matrix
    x = np.linalg.solve(I - A_t, omega_PCE_monthly[t])
    col_sums = A_t.to_numpy().sum(axis=0)
    omega_VA_monthly[t] = x @ np.diag(1 - col_sums)

In [35]:
# quick check on variables greater than 0
for date in omega_VA_monthly:
    bad = omega_VA_monthly[date] < 0 # lots of 0s from products that have not entered yet/exited
    if bad.any():
        print(omega_VA_monthly[date][bad])

18. pi_CPI,t

In [36]:
for t in pi_CPI_monthly.keys():
    pi_CPI_check = omega_VA_monthly[t].T @ (pi_VA_monthly[t] - 1)
    # assert np.isclose(pi_CPI_monthly[t], pi_CPI_check)

    difference_test = pi_CPI_monthly[t] - pi_CPI_check
    bad = abs(difference_test) > 0.0001 
    if bad.any():
        print(difference_test[bad])

    # code is all correct, check tmrw ***


[0.00029681]
[0.00161393]
[0.00111152]
[0.00135677]
[0.00054532]
[0.00100085]
[0.00017854]
[0.0012641]
[0.00098933]
[0.00066501]
[0.00080767]
[0.00076933]
[0.00043948]
[0.00148044]
[0.001066]
[0.00151639]
[0.00065797]
[0.00107238]
[0.00104352]
[0.00098242]
[0.00159619]
[0.00112119]
[0.00079049]
[0.00222696]
[0.00235409]
[0.00222218]
[0.00232023]
[0.00088831]
[0.00142314]
[0.00284578]
[0.00249407]
[0.00308506]
[0.00293879]
[0.00294038]
[0.00328786]
[0.00295925]
[0.00390525]
[0.00371162]
[0.00343824]
[0.00354839]
[0.00323481]
[0.00289474]
[0.00387138]
[0.00271074]
[0.00334399]
[0.0028248]
[0.0017173]
[0.00335295]
[0.00168744]
[0.00266666]
[0.00245809]
[0.00380685]
[0.00579956]
[0.00547241]
[0.00619607]
[0.0096492]
[0.00889988]
[0.00835098]
[0.00787124]
[0.00563625]
[0.00370564]
[0.00590407]
[0.00505868]
[0.00350128]
[0.00251857]
[0.00480469]
[0.00499817]
[0.00551838]
[0.00525904]
[0.00456514]
[0.00433003]
[0.00487804]
[0.00615541]
[0.00519335]
[0.00587019]
[0.00630298]
[0.01127881]
[0.00